## Linear probe: $h \approx \pi\,W_{\mathrm{task}} + x\,W_{\mathrm{tok}} + (\text{nuisance}) + b$

### Model

For each sequence $n$ and position $t$ in a chosen set of fit positions, we solve the multivariate OLS problem

$$
h_{n,t}
\;=\;
\underbrace{\pi_{n,t}}_{\in\,\mathbb{R}^{K}}\,W_{\mathrm{task}}
\;+\;
\underbrace{\mathrm{onehot}(x_{n,t})}_{\in\,\mathbb{R}^{V}}\,W_{\mathrm{tok}}
\;+\;
\underbrace{g_{n,t}}_{\text{logit}}\,W_{\mathrm{logit}}
\;+\;
\underbrace{\mathrm{onehot}(\mathrm{pos}(t))}_{\text{position}}\,W_{\mathrm{pos}}
\;+\;b
\;+\;\varepsilon_{n,t},
$$

where $h_{n,t}\in\mathbb{R}^D$ is the hidden state, $\pi_{n,t}$ is the Bayesian posterior over tasks, $x_{n,t}$ is the current token, $g_{n,t}$ is the model's output logit vector, and the position one-hot captures position-dependent mean shifts (e.g. from RoPE).
Optional feature groups (logit, position) can be enabled or disabled via `include_logit` and `include_position_bias`.

### Dummy-variable trap

When the probe is fitted with **major tasks only** (`sample_mode="major"`), the posterior over the $K$ major tasks satisfies

$$
\pi_1 + \pi_2 + \cdots + \pi_K \;\approx\; 1
$$

for most positions (the Bayesian posterior concentrates on the true major task quickly). Since the OLS design matrix also includes an intercept column $\mathbf{1}$, the augmented matrix $[X \mid \mathbf{1}]$ becomes **near-rank-deficient**: the ones column is approximately a linear combination of the posterior columns. This is the classical *dummy-variable trap* and causes:

- **Huge condition number** ($\kappa \sim 10^7$), making $W_{\mathrm{task}}$ numerically unstable.
- **Gauge ambiguity**: any constant vector $c\in\mathbb{R}^D$ can be added to every row of $W_{\mathrm{task}}$ and subtracted from $b$ without changing predictions, since $\pi\,(W + c) + (b - c) = \pi\,W + b$ when $\sum_k \pi_k = 1$.

### Fix 1 (primary): Minor-task anchoring

When minor tasks are available, the code automatically mixes in additional **train-mode sequences** (which include minor-task sequences) into the OLS fit. On a minor-task sequence, the truncated posterior $(\pi_1, \ldots, \pi_K)$ over major tasks does **not** sum to 1 — some posterior mass goes to the minor task(s). These "anchor" samples break the perfect collinearity between the posterior columns and the intercept, making the design matrix full-rank.

Concretely, on minor-task sequences:
$$
\sum_{k=1}^{K} \pi_k \;<\; 1,
\quad\text{since}\quad
\sum_{k=1}^{K} \pi_k + \sum_{\text{minor}} \pi_j = 1.
$$

This approach:
- Retains all $K$ posterior columns — no arbitrary gauge choice needed.
- Makes $W_{\mathrm{task}}[k]$ uniquely determined (not just up to a constant).
- The centered task vectors $\widetilde{W} = W_{\mathrm{task}} - \bar{w}$ are identical to the column-dropping approach.

Controlled by `anchor_minor_samples` (default: 20% of `n_samples`).

### Fix 2 (fallback): Automatic column dropping

When **no minor tasks exist** (e.g. `n_minor=-1`), anchoring is impossible. The code then detects when $\sum_k \pi_k \approx 1$ (within tolerance $10^{-4}$) and **drops the last posterior column** before fitting:

$$
h_{n,t}
= [\pi_1,\ldots,\pi_{K-1}]\,W_{\mathrm{reduced}}
+ \cdots + b'
+ \varepsilon_{n,t},
$$

where $W_{\mathrm{reduced}}[k] = W_{\mathrm{task}}[k] - W_{\mathrm{task}}[K]$ and $b' = b + W_{\mathrm{task}}[K]$. After fitting, the full $K$-row $W_{\mathrm{task}}$ is reconstructed by setting $W_{\mathrm{task}}[K] = 0$ (a gauge choice). The centered task vectors $\widetilde{W}$ are gauge-invariant.

The same dummy-variable trap applies to the **token** and **position** one-hot groups (each row sums to 1, collinear with the intercept). The code always drops the last column from each one-hot group before fitting and reconstructs full-sized weight matrices afterwards.

---

### `train_linear_hidden_predictor` — parameter reference

#### Data collection

| Parameter | Type | Default | Description |
|---|---|---|---|
| `exp_name` | `str` | — | Experiment name (from `get_exp_name`). Identifies the trained model and sampler. |
| `layer` | `int` | — | Which transformer layer's hidden states to probe (0-indexed). |
| `B` | `int` | `64` | Batch size: number of sequences per forward pass. Does **not** affect total dataset size. |
| `n_samples` | `int` | `1000` | Total number of sequences to generate for fitting. Split into `ceil(n_samples / B)` batches. |
| `step` | `int` or `None` | `None` | Checkpoint step to load. `None` → last checkpoint (`num_epochs`). |
| `n_minor` | `int` or `None` | `None` | Cap on the number of minor tasks to include. `None` → all available; `-1` → zero (major only). |
| `positions` | list or `None` | `None` | Which sequence positions to include in the fit (0-indexed). `None` → first 10. Multiple positions are pooled into a single OLS after adding position-bias features. |
| `sample_mode` | `str` | `"train"` | Sampling mode passed to `sampler.generate(mode=...)`. `"train"` = mix of major + minor (controlled by `p_minor`); `"major"` = major tasks only; `"minor"` = minor tasks only. |
| `uniform_sampling` | `bool` | `True` | When `True` and minor tasks exist, sets `p_minor` so that each task (major or minor) is sampled with equal probability. |
| `device` | `str` or `None` | `None` | Device override (e.g. `"cuda:0"`). `None` → use config default. |
| `verbose` | `bool` | `False` | Print progress messages during data collection. |

#### Fitting

| Parameter | Type | Default | Description |
|---|---|---|---|
| `include_position_bias` | `bool` | `True` | Add one-hot position features to the design matrix (one column per position, last dropped). Captures position-dependent mean shifts from RoPE / positional encoding. Required when pooling multiple positions. |
| `include_logit` | `bool` | `True` | Add the model's output logit vector $g_{n,t}$ as extra regressors. Useful for Latent Markov (logits correlate with the posterior); less so for Coin (logits ≈ linear in posterior). |
| `use_log_posterior` | `bool` | `False` | Use $\log(\pi + \varepsilon)$ instead of $\pi$ as the posterior features. Disables the automatic sum-to-1 column-dropping (log-posteriors don't sum to 1). |
| `validation_split` | `float` | `0.2` | Fraction of sequences held out for validation. The split is on the **sequence** axis (all positions from a sequence go to the same split), preventing information leakage across correlated positions. |
| `skip_baselines` | `bool` | `False` | Skip heavier diagnostics: MLP nonlinearity check, subspace angle computation, per-sample cosine analysis. Speeds up the fit when only R² and weights are needed. |

#### Dummy-variable trap handling

| Parameter | Type | Default | Description |
|---|---|---|---|
| `anchor_minor_samples` | `int` or `None` | `None` | Number of additional train-mode (mixed major+minor) sequences to generate when `sample_mode="major"` and minor tasks exist. These anchor samples break the posterior's sum-to-1 collinearity. `None` → 20% of `n_samples`. Set to `0` to disable anchoring. |

#### Output control

| Parameter | Type | Default | Description |
|---|---|---|---|
| `print_summary` | `bool` | `True` | Print the formatted fit summary table after fitting (R², partial R², F-tests, condition number, VIF, subspace angles). |

### Return dict — key fields

| Key | Shape / Type | Description |
|---|---|---|
| `model_weight` | `(K, D)` | $W_{\mathrm{task}}$: task weight matrix. Row $k$ is the direction in hidden space associated with posterior component $\pi_k$. |
| `model_bias` | `(D,)` | Intercept $b$. |
| `token_weight` | `(V, D)` | $W_{\mathrm{tok}}$: token weight matrix. Row $v$ is the effect of token $v$ on the hidden state. |
| `logit_weight` | `(V_{\mathrm{logit}}, D)` or `(0, D)` | $W_{\mathrm{logit}}$: logit weight matrix. Empty if `include_logit=False`. |
| `position_weight` | `(P, D)` or `None` | $W_{\mathrm{pos}}$: position weight matrix. `None` if `include_position_bias=False` or single position. |
| `val_r2` | `float` | Joint model validation $R^2$. |
| `diagnostics` | `dict` | Partial $R^2$, F-tests, condition number, VIF, GVIF, pairwise $R^2$. |
| `geometry` | `dict` or `None` | Subspace angles between task and token weight spaces, per-sample cosine similarity. `None` if `skip_baselines=True`. |
| `_internals` | `dict` | Per-model-subset fit statistics (`joint_s`, `pi_s`, `tok_s`, etc.) for the print summary. |

---

### MLP baseline (linearity check)

When `skip_baselines=False`, the code fits a **2-layer MLP** on the same design matrix $X_{\mathrm{joint}}$ (posterior + token + logit + position features) to predict the hidden state $h$, and reports its validation $R^2$ alongside the linear $R^2$.

**Architecture:**

$$
f_{\mathrm{MLP}}(x) = W_2 \,\mathrm{ReLU}(W_1 x + b_1) + b_2,
\qquad W_1 \in \mathbb{R}^{128 \times p},\; W_2 \in \mathbb{R}^{D \times 128},
$$

where $p = d_{\mathrm{post}} + d_{\mathrm{tok}} + d_{\mathrm{logit}} + d_{\mathrm{pos}}$ is the total number of input features (after dummy-variable column drops) and $D$ is the hidden state dimension. The hidden layer width is 128.

**Training:**
- **Optimizer:** Adam, learning rate $10^{-3}$
- **Epochs:** 200
- **Batch size:** 4096 (mini-batch SGD with random permutation each epoch)
- **Loss:** MSE $\|h - f_{\mathrm{MLP}}(x)\|^2$, averaged over the batch

**Evaluation:**
- Validation $R^2 = 1 - \mathrm{SS}_{\mathrm{res}} / \mathrm{SS}_{\mathrm{tot}}$ on the held-out set (same split as OLS)

**Interpretation:**
- The **linear gap** = MLP $R^2$ $-$ OLS $R^2$ measures how much predictive power is gained by allowing nonlinear feature interactions. A small gap (e.g. $<0.05$) suggests the linear model captures the essential structure. A large gap suggests the hidden state encodes task/token information in a nonlinear manner that a linear probe cannot recover.
- The MLP uses the *same features* as the linear probe — it cannot discover new information, only exploit nonlinear combinations of the existing features.

In [ ]:
from icl.latent_markov.analysis.probes import train_linear_hidden_predictor
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", -1)
res = train_linear_hidden_predictor(
    exp_name, 
    layer=3,
    sample_mode="major",
    n_samples=2**13,
    positions=range(30),
    include_position_bias=False,
    include_logit=False,
    )

## Trajectory projection plot: `traj_post_posterior_projection_plot`

This function answers the question: *does the model's hidden state track the Bayesian posterior over tasks as the sequence unfolds?*

For each task $k$, it extracts the hidden-state trajectory $h^{(\ell)}(t)$ at every position $t$, estimates a "projected posterior" $\lambda(t)$ by projecting $h$ onto the fitted task-weight basis, and overlays the oracle Bayesian posterior $P(Z = k \mid x_{1:t})$.

---

### Step 1: Probe fitting

Calls `train_linear_hidden_predictor` with `sample_mode="major"` on the chosen layer $\ell$ to obtain:

| Weight | Shape | Description |
|---|---|---|
| $W_{\mathrm{task}}$ | $(K, D)$ | Task weight matrix (row $k$ = direction for task $k$) |
| $W_{\mathrm{tok}}$ | $(V, D)$ | Token weight matrix |
| $W_{\mathrm{logit}}$ | $(V_{\mathrm{logit}}, D)$ | Logit weight matrix (empty if `fit_include_logit=False`) |
| $W_{\mathrm{pos}}$ | $(P, D)$ | Position weight matrix (empty if `fit_include_position_bias=False`) |
| $b$ | $(D,)$ | Intercept |

These are the same OLS weights described in the probe section above.

### Step 2: Data generation

Generates $B$ sequences per task (major tasks + OOD tasks) via `_get_hiddens_at_real_positions`. Collects hidden states $h^{(\ell)}(t)$ of shape $(K_{\mathrm{total}}, T, B, D)$ at **all** sequence positions for the selected layer, where $K_{\mathrm{total}}$ includes both major and OOD tasks.

### Step 3: Nuisance subtraction

For each task $k$ and each sequence $n$, subtracts the fitted nuisance contributions from the hidden state:

$$
h_{\mathrm{adj}}^{(n,t)}
\;=\;
h^{(n,t)}
\;-\; b
\;-\; \underbrace{W_{\mathrm{tok}}[x_{n,t}]}_{\text{token}}
\;-\; \underbrace{g_{n,t}\,W_{\mathrm{logit}}}_{\text{logit}}
\;-\; \underbrace{W_{\mathrm{pos}}[\mathrm{pos}(t)]}_{\text{position}}
$$

where:
- **Token nuisance**: row lookup $W_{\mathrm{tok}}[x_{n,t}]$ by the actual token at position $t$.
- **Logit nuisance**: $g_{n,t} \cdot W_{\mathrm{logit}}$, the model's output logit vector times the logit weight matrix. Only computed if `fit_include_logit=True` (requires reloading the model for a forward pass).
- **Position nuisance**: row lookup $W_{\mathrm{pos}}[\mathrm{pos}(t)]$ for positions in the fit set; zero for positions outside the fit set.

After this step, the residual should be approximately $h_{\mathrm{adj}} \approx \pi \, W_{\mathrm{task}} + \varepsilon$.

### Step 4: Lambda estimation via `estimate_lambda_with_r2`

Projects $h_{\mathrm{adj}}$ onto the task-weight basis to recover a "projected posterior" $\lambda(t) \in \mathbb{R}^K$.

**Centered mode** (`center_lambda=True`, default):

1. Compute centered task vectors: $\widetilde{W} = W_{\mathrm{task}} - \bar{w}$, where $\bar{w} = \frac{1}{K}\sum_k W_{\mathrm{task}}[k]$.
2. Further subtract the mean: $h' = h_{\mathrm{adj}} - \bar{w}$.
3. Drop the last column of $\widetilde{W}^{\!\top}$ (since $\widetilde{W}$ rows sum to zero, the last column is redundant).
4. Solve the $D$-dimensional OLS problem treating coordinates as observations:
$$
\hat{\lambda}_{1:K-1} = \arg\min_{\lambda} \| h' - \widetilde{W}_{:,1:K-1}^{\!\top}\,\lambda \|^2
$$
5. Redistribute residual mass: $\hat{\lambda}_k \mathrel{+}= \frac{1 - \sum_{j<K} \hat{\lambda}_j}{K}$ for all $k$, so that $\sum_k \hat{\lambda}_k = 1$.
6. Project $\hat{\lambda}$ onto the probability simplex (Euclidean projection, ensuring $\lambda_k \geq 0$ and $\sum_k \lambda_k = 1$).

**Unconstrained mode** (`center_lambda=False`):

1. Solve $\hat{\lambda} = (W^{\!\top} W)^{+} W^{\!\top} h_{\mathrm{adj}}$ using `torch.linalg.lstsq`.
2. Clip each $\hat{\lambda}_k$ to $[0, 1]$.

### Step 5: Oracle posterior computation

Computes the Bayesian posterior $P(Z = k \mid x_{1:t})$ using `task_posterior_over_time` with `include_minor=False`. This gives the ground-truth posterior over **major tasks only**, shape $(B, T, K_{\mathrm{major}})$.

### Step 6: Plotting

For each task $k$ in `task_ids`:
- Plots $\lambda_k(t)$ (projected posterior) and $P(Z = k \mid x_{1:t})$ (oracle posterior) as **mean $\pm$ std** bands over $B$ sequences.
- Overlays `n_individual` individual traces for both quantities.
- Each row = one task. Major tasks and OOD tasks are split into **separate figures**.
- Colors: the three corners of the simplex are colored by `corner_colors`; for major tasks, $\lambda_k$ is colored by the task's corner color.

**Interpretation**: if the model's hidden state linearly encodes the Bayesian posterior, then $\lambda_k(t)$ should closely track $P(Z = k \mid x_{1:t})$. Deviations indicate either nonlinear encoding, information loss, or systematic biases in the probe fit.

---

### `traj_post_posterior_projection_plot` — parameter reference

| Parameter | Type | Default | Description |
|---|---|---|---|
| `exp_name` | `str` | — | Experiment name (from `get_exp_name`). |
| `layer_index` | `int` or `None` | `None` | Layer to probe. `None` → last layer. |
| `task_ids` | `list[int]` or `None` | `None` | Which tasks to show as rows. 0..K-1 = major; $\geq K$ = OOD. Default `[0, 1, 2]`. |
| `n_individual` | `int` | `10` | Number of individual sample traces shown behind the mean band. |
| `B` | `int` | `64` | Number of sequences generated per task. |
| `step` | `int` or `None` | `None` | Checkpoint step. `None` → last. |
| `fit_n_samples` | `int` | `5000` | Total sequences for the probe OLS fit. |
| `fit_positions` | `list` or `None` | `None` | Positions used for probe fitting. `None` → positions 100 to seq_len. |
| `fit_include_position_bias` | `bool` | `True` | Include one-hot position features in the probe. |
| `fit_include_logit` | `bool` | `True` | Include logit features in the probe. |
| `n_ood` | `int` or `None` | `None` | Number of OOD tasks to generate. Auto-determined from `task_ids` if not given. |
| `center_lambda` | `bool` | `True` | If `True`, project $\lambda$ onto the probability simplex. If `False`, clip to $[0,1]$. |
| `figsize` | `tuple` or `None` | `None` | Override figure size. |
| `show` | `bool` | `True` | Display the figure. |
| `corner_colors` | `tuple` | `("#1f77b4", "#2ca02c", "#d62728")` | RGB hex colors for the $K=3$ simplex corners. |

## Direct posterior injection (no OOD sequences needed)

### Motivation

The OOD-based `intervene_inject_posterior` requires generating out-of-distribution
sequences, computing their misspecified Bayesian posteriors, and filtering for
non-convergent samples. The **direct** variant below bypasses all of that:

1. **Input**: sequences sampled from the *major* tasks (no OOD sampler).
2. **$\alpha$ sampling**: draw $\alpha \sim \mathrm{Dirichlet}(1,\dots,1)$ independently
   per sample — one $\alpha$ per sample, constant across all positions.
3. **Injection**: at the target layer, replace the task-subspace component of
   every hidden state:
   $$h'_t = h_t - h_t\,P_{\mathrm{task}} + \alpha\,W_{\mathrm{task}}$$
4. **Target output**: the $\alpha$-weighted Markov mixture
   $q(s_{t+1} \mid s_t) = \sum_k \alpha_k\,P_k(s_{t+1} \mid s_t)$.
   Unlike the coin task, the target here depends on position through $s_t$.
5. **Metrics** (per position):
   - $\mathrm{KL}(\text{baseline} \| q)$ — how far the unmodified model is from the mixture.
   - $\mathrm{KL}(\text{injected} \| q)$ — how well the injection recovers the mixture.
   - $\mathrm{KL}(\text{mode} \| q)$ — gap between using only the MAP task
     $P_{k^*}$ and the full mixture.

### Why this is cleaner

- $\alpha$ is **controlled**, not inferred from noisy data — no posterior
  convergence issue.
- A large drop from baseline KL to injected KL, with injected KL well
  *below* the mode KL, is direct causal evidence that the task subspace
  linearly encodes mixture weights.

In [ ]:
from icl.latent_markov.analysis.interventions import (
    intervene_direct_injection,
    plot_inject_posterior_per_position,
)
from icl.utils.unified_interface import get_exp_name
exp_name = get_exp_name("latent", k=-1)
result = intervene_direct_injection(
    exp_name=exp_name,
    layer=5,
    B=64,
    n_samples=2**14,
    eval_positions=list(range(60)),
    fit_n_samples=2**13,
    fit_positions=list(range(30)),
    center_task_vecs=True,
    dirichlet_alpha=0.2,
)
fig = plot_inject_posterior_per_position(result)

In [ ]:
from icl.latent_markov.analysis.interventions import (
    intervene_direct_injection,
    plot_inject_posterior_per_position,
)
from icl.utils.unified_interface import get_exp_name
exp_name = get_exp_name("latent", k=-1)
result = intervene_direct_injection(
    exp_name=exp_name,
    layer=3,
    B=64,
    n_samples=2**14,
    eval_positions=list(range(60)),
    fit_n_samples=2**13,
    fit_positions=list(range(30)),
    center_task_vecs=True,
    dirichlet_alpha=1.0,
)
fig = plot_inject_posterior_per_position(result)

In [ ]:
from icl.latent_markov.analysis import traj_post_posterior_projection_plot
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", k=8)

out = traj_post_posterior_projection_plot(
    exp_name,
    layer_index=3,
    task_ids=range(3),   # major task
    fit_positions=list(range(192)),
    B=64,
    fit_include_position_bias=False,
    fit_include_logit=False,
)
# Or task_id=5 for OOD (λ vs misspecified posterior)

In [ ]:
from icl.latent_markov.analysis import traj_cellmean_projection_plot
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", k=-1)
result = traj_cellmean_projection_plot(exp_name, layer_index=4, annotate_agreement=False,estimation_B=1024)

In [ ]:
from icl.latent_markov.analysis import traj_cellmean_projection_plot
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", k=8)
result = traj_cellmean_projection_plot(exp_name, layer_index=3, annotate_agreement=False,estimation_B=1024)

In [ ]:
from icl.latent_markov.analysis import plot_lambda_posterior_agreement
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", k=-1)
out = plot_lambda_posterior_agreement(
    exp_name=exp_name,      # e.g. get_exp_name("latent", k)
    layer_index=3,          # None -> last layer
    n_ood=30,
    B=64,
    fit_n_samples=5000,
    fit_positions=list(range(3, 30)),
    fit_include_position_bias=False,
    fit_include_logit=False,
    max_position=30,
    figsize=(18, 5),
    title="",
    show=True,
)

# Returned keys include:
# out["tv_major"], out["tv_ood"], out["cos_major"], out["cos_ood"],
# out["rcorr_major"], out["rcorr_ood"], out["positions"]

In [ ]:
from icl.latent_markov.analysis import traj_post_posterior_projection_plot
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", k=-1)

out = traj_post_posterior_projection_plot(
    exp_name,
    layer_index=3,
    task_ids=range(5),   # major task
    B=64,
    fit_positions=list(range(96, 192)),
    fit_include_position_bias=False,
    fit_include_logit=False,
)
# Or task_id=5 for OOD (λ vs misspecified posterior)

In [ ]:
from icl.utils.unified_plot import plot_task_vector_geometry
from icl.utils.unified_interface import get_exp_name

exp_name = get_exp_name("latent", k=-1)
result = plot_task_vector_geometry("latent", exp_name, layer_index=3, B=1024, marker_every=1, plot_positions=range(1, 40))